# Phase 0 — Pipeline complet : Preprocessing -> Annotation GFW -> Benchmark

## Sequence unique, executee entierement sur Colab

```
[1] Setup + montage Drive (lecture seule .SAFE)
[2] Verification CRS -- bloquant, jamais de reconstruction devinee
[3] Pour chaque scene :
      3a. Lire bbox globale (metadonnees seules, pas de pixels)
      3b. GFW : SAR detections + AIS + dark vessels sur cette bbox
      3c. Selection stratifiee des tiles (GFW + echantillon vide)
      3d. Traitement A/B/C/D SEULEMENT sur les tiles selectionnees
      3e. Ecriture atomique metadata.json (tiles + labels + geo_bbox)
[4] Sanity check global
[5] Benchmark mAP / Precision / Recall par pipeline vs labels GFW
[6] Test KS inter-pipelines (A/B/C/D) -- MRSSD reel non disponible, limitation documentee
[7] Export final reduit
```

## Ce que corrige cette version par rapport aux tentatives precedentes

| Defaut precedent | Correction ici |
|---|---|
| `geo_bbox` parfois reconstruit par approximation (`abs(px_y) > 90` -> centre arbitraire 32.0/-9.0) | CRS verifie explicitement a l'ouverture du GeoTIFF. Si non-WGS84 : reprojection formelle ou echec **explicite** de la scene. Aucune coordonnee devinee. |
| Toutes les tiles calculees puis 95% jetees (68 Go) | GFW interroge **avant** le tiling complet, sur la bbox globale de la scene. Seules les tiles pertinentes sont calculees. |
| `annotate_scene_with_ais` : mauvais endpoint (`vessel-positions`), pas de SAR detections, `except: return []` muet | Reimplementation complete : `/v3/events` avec `public-global-sar-vessel-detections:latest`, fallback GET->POST, logging explicite des echecs. |
| `run_pipeline_benchmark` : test KS contre une distribution **synthetique**, aucun mAP reel | Calcul reel de Precision/Recall/mAP@0.5 a partir des labels GFW projetes en bounding boxes. Test KS limite a une comparaison **inter-pipelines** -- comparaison a MRSSD reel non faite ici faute d'acces confirme au dataset. |

## Prerequis avant de lancer

- Les 12 `.SAFE` marocains 2025 sont dans
  `/content/drive/MyDrive/Ksf_maritime_experiments/scenes/`
- Un `GFW_API_TOKEN` valide (renseigne cellule 4)
- **Executer d'abord sur une seule scene** avant de lancer le batch complet.


In [ ]:
# Cellule 1 -- Dependances
!pip install -q rasterio scipy numpy tqdm matplotlib pillow psutil httpx


In [ ]:
# Cellule 2 -- Montage Drive (lecture seule pour les .SAFE sources)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Cellule 3 -- Configuration et constantes

import os
import time
import json
import gc
import zipfile
import logging
import math
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import rasterio
from rasterio.warp import transform as warp_transform
from rasterio.windows import Window
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import uniform_filter
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import psutil
import httpx

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("phase0")

ROOT_SAFE_DIR = Path("/content/drive/MyDrive/Ksf_maritime_experiments/scenes/")

WORK_DIR = Path("/content/phase0")
TILES_DIR = WORK_DIR / "data" / "tiles"
RESULTS_DIR = WORK_DIR / "data" / "results"
VIZ_DIR = WORK_DIR / "data" / "viz"
ANNOT_DIR = WORK_DIR / "data" / "annotations"

for d in [TILES_DIR, RESULTS_DIR, VIZ_DIR, ANNOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TILE_SIZE = 512
OVERLAP = 0.5
NODATA_THRESHOLD = 0.30
DB_MIN, DB_MAX = -30.0, 0.0
PIPELINES = ["A", "B", "C", "D"]
POLARIZATION = "vv"

GFW_MARGIN_DEG = 0.01
N_EMPTY_TILES_PER_SCENE = 80
MAX_TILES_PER_SCENE_HARD_CAP = 600

AIS_SAR_MATCH_DISTANCE_M = 500.0

def get_ram_mb() -> float:
    return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)

def haversine_m(lat1, lon1, lat2, lon2) -> float:
    R = 6371000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlmb/2)**2
    return 2 * R * math.asin(math.sqrt(a))

print(f"Scenes sources : {ROOT_SAFE_DIR}")
print(f"Sortie locale  : {WORK_DIR}")
print(f"RAM actuelle   : {get_ram_mb():.0f} MB")


In [ ]:
# Cellule 4 -- Configuration GFW

GFW_API_TOKEN = ""  # <-- renseigner avant execution
GFW_BASE_URL = "https://gateway.api.globalfishingwatch.org/v3"

assert GFW_API_TOKEN, "GFW_API_TOKEN vide -- renseigner le token avant de continuer."

_gfw_client = httpx.Client(
    timeout=30.0,
    headers={"Authorization": f"Bearer {GFW_API_TOKEN}"},
)

print("Client GFW configure.")


In [ ]:
# Cellule 5 -- Verification CRS -- BLOQUANTE, remplace toute reconstruction devinee
#
# Principe : on ne devine jamais une coordonnee a partir de "ca ressemble a
# un pixel". On verifie le systeme de reference du GeoTIFF explicitement.
# Si ce n'est pas WGS84 (EPSG:4326), on reprojette formellement ou on fait
# echouer la scene -- jamais de coordonnee fabriquee.

def find_safe_files(safe_path: str, polarization: str = "vv") -> Dict[str, Optional[str]]:
    pol = polarization.lower()
    safe_dir = Path(safe_path)
    m_dir = safe_dir / "measurement"
    tiff_candidates = list(m_dir.glob(f"*-{pol}-*-cog.tiff")) or list(m_dir.glob(f"*-{pol}-*.tiff"))
    if not tiff_candidates:
        raise FileNotFoundError(f"Aucun TIFF {pol} trouve dans {m_dir}")
    cal_dir = safe_dir / "annotation" / "calibration"
    cal_candidates = list(cal_dir.glob(f"calibration-*-{pol}-*.xml"))
    if not cal_candidates:
        raise FileNotFoundError(f"Aucun XML de calibration {pol} trouve dans {cal_dir}")
    noise_candidates = list(cal_dir.glob(f"noise-*-{pol}-*.xml"))
    return {
        "tiff": str(tiff_candidates[0]),
        "calibration": str(cal_candidates[0]),
        "noise": str(noise_candidates[0]) if noise_candidates else None,
    }


def verify_crs(tiff_path: str) -> dict:
    """
    Verifie le CRS d'un GeoTIFF Sentinel-1. Retourne un rapport explicite.
    NE DEVINE RIEN -- soit le CRS est WGS84 (usage direct de src.xy()),
    soit il faut reprojeter (warp_transform), soit on echoue.
    """
    with rasterio.open(tiff_path) as src:
        crs = src.crs
        is_wgs84 = crs is not None and crs.to_epsg() == 4326
        return {
            "crs": str(crs),
            "epsg": crs.to_epsg() if crs else None,
            "is_wgs84": is_wgs84,
            "width": src.width,
            "height": src.height,
            "bounds": tuple(src.bounds),
        }


def pixel_to_latlon_verified(src, row: int, col: int) -> Tuple[float, float]:
    """
    Convertit (row, col) -> (lat, lon), avec reprojection explicite si le
    CRS source n'est pas WGS84. Leve une exception si le resultat sort des
    bornes valides -- pas de coordonnee fabriquee en silence.
    """
    x, y = src.xy(row, col)
    if src.crs is not None and src.crs.to_epsg() != 4326:
        lon_arr, lat_arr = warp_transform(src.crs, "EPSG:4326", [x], [y])
        lon, lat = lon_arr[0], lat_arr[0]
    else:
        lon, lat = x, y

    if not (-90.0 <= lat <= 90.0) or not (-180.0 <= lon <= 180.0):
        raise ValueError(
            f"Coordonnee hors bornes apres transformation: lat={lat}, lon={lon}. "
            f"CRS source={src.crs}. Scene rejetee -- pas de reconstruction."
        )
    return lat, lon


_test_scenes = sorted(ROOT_SAFE_DIR.glob("*.SAFE"))
if not _test_scenes:
    raise FileNotFoundError(f"Aucune scene .SAFE dans {ROOT_SAFE_DIR} -- verifier le montage Drive.")

_test_files = find_safe_files(str(_test_scenes[0]))
_crs_report = verify_crs(_test_files["tiff"])

print(f"Scene testee : {_test_scenes[0].name}")
print(f"CRS detecte  : {_crs_report['crs']} (EPSG:{_crs_report['epsg']})")
print(f"WGS84 direct : {_crs_report['is_wgs84']}")
print(f"Bounds natifs: {_crs_report['bounds']}")

if not _crs_report["is_wgs84"]:
    logger.warning(
        "CRS non-WGS84 detecte -- reprojection activee automatiquement via "
        "rasterio.warp.transform dans pixel_to_latlon_verified(). "
        "Verifier manuellement un echantillon de coordonnees avant le batch complet."
    )
else:
    logger.info("CRS WGS84 confirme -- usage direct de src.xy(), aucune reprojection necessaire.")


In [ ]:
# Cellule 6 -- Calibration LUT sparse

class SparseCalibrationLUT:
    def __init__(self, cal_path: str, noise_path: Optional[str]):
        self.sigma_values, self.sigma_lines, self.sigma_pixels = self._parse_cal(cal_path)
        self.noise_values = self.noise_lines = self.noise_pixels = None
        if noise_path:
            try:
                parsed = self._parse_noise(noise_path)
                if parsed:
                    self.noise_values, self.noise_lines, self.noise_pixels = parsed
            except Exception as e:
                logger.warning(f"Noise LUT non exploitable ({e}) -- bruit thermique ignore.")

    def _parse_cal(self, path: str):
        root = ET.parse(path).getroot()
        vectors = root.findall(".//calibrationVector")
        if not vectors:
            raise ValueError(f"Aucun calibrationVector dans {path}")
        pixel_indices = np.array([int(p) for p in vectors[0].find("pixel").text.split()])
        lines, sigma_values = [], []
        for vec in vectors:
            lines.append(int(vec.find("line").text))
            sigma_values.append([float(v) for v in vec.find("sigmaNought").text.split()])
        return np.array(sigma_values), np.array(lines), pixel_indices

    def _parse_noise(self, path: str):
        root = ET.parse(path).getroot()
        vectors = root.findall(".//noiseRangeLut") or root.findall(".//noiseLut")
        if not vectors:
            return None
        pixel_node = vectors[0].find("pixel")
        if pixel_node is None or pixel_node.text is None:
            return None
        pixel_indices = np.array([int(p) for p in pixel_node.text.split()])
        lines, noise_values = [], []
        for vec in vectors:
            line_node = vec.find("line")
            if line_node is None:
                continue
            node = vec.find("noiseLut")
            if node is None:
                node = vec.find("noiseRangeLut")
            if node is not None and node.text is not None:
                lines.append(int(line_node.text))
                noise_values.append([float(v) for v in node.text.split()])
        if not noise_values:
            return None
        return np.array(noise_values), np.array(lines), pixel_indices

    def get_window_lut(self, lut_type: str, window_coords, out_shape) -> Optional[np.ndarray]:
        vals = self.sigma_values if lut_type == "sigma" else self.noise_values
        lines = self.sigma_lines if lut_type == "sigma" else self.noise_lines
        pixs = self.sigma_pixels if lut_type == "sigma" else self.noise_pixels
        if vals is None or lines is None or pixs is None:
            return None
        row_start, row_end, col_start, col_end = window_coords
        interpolator = RegularGridInterpolator(
            (lines, pixs), vals, method="linear", bounds_error=False, fill_value=None
        )
        lg, pg = np.meshgrid(
            np.arange(row_start, row_end), np.arange(col_start, col_end), indexing="ij"
        )
        pts = np.column_stack([lg.ravel(), pg.ravel()])
        return interpolator(pts).reshape(out_shape).astype(np.float32)

In [ ]:
# Cellule 7 -- Les 4 pipelines A/B/C/D

def apply_lee_filter(data: np.ndarray, kernel_size: int = 5) -> np.ndarray:
    local_mean = uniform_filter(data, size=kernel_size)
    local_var = uniform_filter(data ** 2, size=kernel_size) - local_mean ** 2
    noise_var = np.mean(np.maximum(local_var, 0))
    weight = np.maximum(local_var - noise_var, 0) / np.maximum(local_var, noise_var + 1e-10)
    return (local_mean + weight * (data - local_mean)).astype(np.float32)


def process_window_pipeline(
    data_uint16: np.ndarray,
    sigma_lut: Optional[np.ndarray],
    noise_lut: Optional[np.ndarray],
    pipeline_type: str,
) -> np.ndarray:
    """
    A -- Raw        : uint16 -> percentile [1,99] -> uint8
    B -- Calibration : uint16 -> sigma0 -> dB -> lineaire [-30,0] -> uint8
    C -- + Lee       : uint16 -> sigma0 -> Lee 5x5 -> dB -> lineaire -> uint8
    D -- + equalize  : uint16 -> sigma0 -> Lee 5x5 -> dB -> min-max local -> uint8
    """
    dn = data_uint16.astype(np.float32)

    if pipeline_type == "A":
        p1, p99 = np.percentile(dn, [1, 99])
        return (np.clip((dn - p1) / (p99 - p1 + 1e-6), 0, 1) * 255).astype(np.uint8)

    if sigma_lut is None:
        raise ValueError("sigma_lut requis pour B/C/D")

    dn2 = np.maximum(dn ** 2 - noise_lut, 0) if noise_lut is not None else dn ** 2
    sigma0 = dn2 / (np.maximum(sigma_lut, 1e-10) ** 2)

    if pipeline_type == "B":
        db = 10 * np.log10(sigma0 + 1e-10)
        return (np.clip((db - DB_MIN) / (DB_MAX - DB_MIN), 0, 1) * 255).astype(np.uint8)

    filtered = apply_lee_filter(sigma0)

    if pipeline_type == "C":
        db = 10 * np.log10(filtered + 1e-10)
        return (np.clip((db - DB_MIN) / (DB_MAX - DB_MIN), 0, 1) * 255).astype(np.uint8)

    if pipeline_type == "D":
        db = 10 * np.log10(filtered + 1e-10)
        d_min, d_max = db.min(), db.max()
        if d_max <= d_min:
            return np.zeros_like(dn, dtype=np.uint8)
        return ((db - d_min) / (d_max - d_min) * 255).astype(np.uint8)

    raise ValueError(f"pipeline_type inconnu: {pipeline_type}")


In [ ]:
# Cellule 8 -- Client GFW : SAR detections + AIS + dark vessels

def _parse_acquisition_time(scene_id: str) -> Optional[str]:
    import re
    m = re.search(r"_(\d{8})T(\d{6})_", scene_id)
    if not m:
        return None
    d, t = m.groups()
    return f"{d[:4]}-{d[4:6]}-{d[6:8]}T{t[:2]}:{t[2:4]}:{t[4:6]}Z"


def get_scene_bbox_from_tiff(tiff_path: str) -> List[float]:
    """
    Retourne la bbox globale [lat_min, lon_min, lat_max, lon_max] de la scene
    a partir des metadonnees du GeoTIFF SEULEMENT -- aucune lecture de pixels.
    """
    with rasterio.open(tiff_path) as src:
        lat1, lon1 = pixel_to_latlon_verified(src, 0, 0)
        lat2, lon2 = pixel_to_latlon_verified(src, src.height, src.width)
    return [min(lat1, lat2), min(lon1, lon2), max(lat1, lat2), max(lon1, lon2)]


def gfw_get_sar_detections(bbox: List[float], date_start: str, date_end: str, limit: int = 500) -> List[dict]:
    """
    GET/POST /v3/events, dataset public-global-sar-vessel-detections:latest.
    """
    lat_min, lon_min, lat_max, lon_max = bbox
    geometry = {
        "type": "Polygon",
        "coordinates": [[
            [lon_min, lat_min], [lon_max, lat_min],
            [lon_max, lat_max], [lon_min, lat_max],
            [lon_min, lat_min],
        ]],
    }

    all_entries = []
    try:
        params = {
            "datasets[0]": "public-global-sar-vessel-detections:latest",
            "start-date": date_start,
            "end-date": date_end,
            "limit": limit,
            "offset": 0,
            "geometry": json.dumps(geometry),
        }
        r = _gfw_client.get(f"{GFW_BASE_URL}/events", params=params)
        logger.info(f"SAR GET /events -> {r.status_code}")
        if r.status_code == 200:
            all_entries = r.json().get("entries", [])
        else:
            logger.warning(f"SAR GET echec ({r.status_code}): {r.text[:300]}")
    except Exception as e:
        logger.warning(f"SAR GET exception: {e}")

    if not all_entries:
        try:
            body = {
                "datasets": ["public-global-sar-vessel-detections:latest"],
                "startDate": date_start,
                "endDate": date_end,
                "geometry": geometry,
                "limit": limit,
            }
            r = _gfw_client.post(f"{GFW_BASE_URL}/events", json=body)
            logger.info(f"SAR POST /events -> {r.status_code}")
            if r.status_code == 200:
                all_entries = r.json().get("entries", [])
            else:
                logger.warning(f"SAR POST echec ({r.status_code}): {r.text[:300]}")
        except Exception as e:
            logger.warning(f"SAR POST exception: {e}")

    results = []
    for entry in all_entries:
        sar = entry.get("sar", {}) or {}
        cat = sar.get("matchedCategory", "unmatched")
        if cat == "infrastructure":
            continue
        pos = entry.get("position", {})
        results.append({
            "id": entry.get("id", ""),
            "lat": pos.get("lat"), "lon": pos.get("lon"),
            "timestamp": entry.get("start", ""),
            "matched_to_ais": cat == "matched",
            "detection_type": cat,
            "confidence": sar.get("confidence"),
            "estimated_length_m": sar.get("length"),
            "label": "AIS_confirmed" if cat == "matched" else "visual_only",
            "source": "sar_detection",
        })
    logger.info(f"SAR detections retenues (hors infra) : {len(results)}")
    return results


def gfw_get_ais_off_events(bbox: List[float], date_start: str, date_end: str, limit: int = 200) -> List[dict]:
    """Dark vessel candidates via public-global-gaps-events:latest."""
    lat_min, lon_min, lat_max, lon_max = bbox
    geometry = {
        "type": "Polygon",
        "coordinates": [[
            [lon_min, lat_min], [lon_max, lat_min],
            [lon_max, lat_max], [lon_min, lat_max],
            [lon_min, lat_min],
        ]],
    }
    try:
        params = {
            "datasets[0]": "public-global-gaps-events:latest",
            "start-date": date_start,
            "end-date": date_end,
            "limit": limit,
            "offset": 0,
            "geometry": json.dumps(geometry),
        }
        r = _gfw_client.get(f"{GFW_BASE_URL}/events", params=params)
        logger.info(f"Gaps GET /events -> {r.status_code}")
        if r.status_code != 200:
            logger.warning(f"Gaps echec ({r.status_code}): {r.text[:300]}")
            return []
        entries = r.json().get("entries", [])
    except Exception as e:
        logger.warning(f"Gaps exception: {e}")
        return []

    results = []
    for entry in entries:
        pos = entry.get("position", {})
        if pos.get("lat") is None:
            continue
        results.append({
            "lat": pos["lat"], "lon": pos["lon"],
            "timestamp_off": entry.get("start", ""),
            "source": "ais_off",
            "label": "dark_vessel_candidate",
        })
    logger.info(f"AIS-off events retenus : {len(results)}")
    return results


def fetch_scene_gfw_annotations(scene_id: str, bbox: List[float]) -> List[dict]:
    """
    Une seule paire d'appels GFW par scene, AVANT le tiling.
    """
    acq_time = _parse_acquisition_time(scene_id)
    if acq_time is None:
        logger.warning(f"{scene_id}: heure d'acquisition non parsable, fenetre GFW = jour civil large")
        date_start = date_end = "2025-01-01"
    else:
        date_start = acq_time[:10]
        date_end = date_start

    sar = gfw_get_sar_detections(bbox, date_start, date_end)
    dark = gfw_get_ais_off_events(bbox, date_start, date_end)

    all_annotations = sar + dark
    if not all_annotations:
        logger.warning(f"{scene_id}: AUCUNE annotation GFW trouvee (SAR ni dark). "
                        f"La scene sera echantillonnee aleatoirement uniquement.")
    return all_annotations


In [ ]:
# Cellule 9 -- Echantillonnage stratifie des tiles (resout le probleme des 68 Go)

def compute_tile_grid_positions(height: int, width: int, tile_size: int, overlap: float) -> List[Tuple[int, int, int, int]]:
    stride = int(tile_size * (1 - overlap))
    return [
        (x, y, min(tile_size, width - x), min(tile_size, height - y))
        for y in range(0, height, stride)
        for x in range(0, width, stride)
    ]


def select_tiles_for_scene(
    src,
    positions: List[Tuple[int, int, int, int]],
    stride: int,
    gfw_annotations: List[dict],
    rng: np.random.Generator,
) -> Tuple[List[dict], Dict[str, List[dict]]]:
    """
    Retourne (tiles_a_traiter, annotations_par_tile).
    """
    candidate_tiles = []
    for (x, y, tw, th) in positions:
        lat1, lon1 = pixel_to_latlon_verified(src, y, x)
        lat2, lon2 = pixel_to_latlon_verified(src, y + th, x + tw)
        geo_bbox = [min(lat1, lat2), min(lon1, lon2), max(lat1, lat2), max(lon1, lon2)]
        t_id = f"r{y // stride:04d}_c{x // stride:04d}"
        candidate_tiles.append({
            "x": x, "y": y, "tw": tw, "th": th,
            "tile_id": t_id,
            "pixel_bbox": [x, y, x + tw, y + th],
            "geo_bbox": geo_bbox,
        })

    tiles_with_gfw = []
    annotations_by_tile: Dict[str, List[dict]] = {}

    for tile in candidate_tiles:
        lat_min, lon_min, lat_max, lon_max = tile["geo_bbox"]
        lat_min_m, lat_max_m = lat_min - GFW_MARGIN_DEG, lat_max + GFW_MARGIN_DEG
        lon_min_m, lon_max_m = lon_min - GFW_MARGIN_DEG, lon_max + GFW_MARGIN_DEG

        matches = [
            a for a in gfw_annotations
            if a.get("lat") is not None
            and lat_min_m <= a["lat"] <= lat_max_m
            and lon_min_m <= a["lon"] <= lon_max_m
        ]
        if matches:
            tile["reason"] = "gfw_annotation"
            tiles_with_gfw.append(tile)
            annotations_by_tile[tile["tile_id"]] = matches

    remaining = [t for t in candidate_tiles if t["tile_id"] not in annotations_by_tile]
    n_empty = min(N_EMPTY_TILES_PER_SCENE, len(remaining))
    if n_empty > 0:
        idx = rng.choice(len(remaining), size=n_empty, replace=False)
        empty_sample = [remaining[i] for i in idx]
    else:
        empty_sample = []
    for t in empty_sample:
        t["reason"] = "empty_control_sample"

    selected = tiles_with_gfw + empty_sample

    if len(selected) > MAX_TILES_PER_SCENE_HARD_CAP:
        logger.warning(
            f"Cap dur atteint ({len(selected)} > {MAX_TILES_PER_SCENE_HARD_CAP}), "
            f"troncature aleatoire -- zone probablement a trafic tres dense."
        )
        idx2 = rng.choice(len(selected), size=MAX_TILES_PER_SCENE_HARD_CAP, replace=False)
        selected = [selected[i] for i in idx2]

    logger.info(
        f"Selection: {len(tiles_with_gfw)} tiles GFW + {len(empty_sample)} tiles vides "
        f"= {len(selected)} / {len(candidate_tiles)} candidates "
        f"({100*len(selected)/max(len(candidate_tiles),1):.1f}%)"
    )
    return selected, annotations_by_tile


In [ ]:
# Cellule 10 -- Traitement complet d'une scene : GFW -> selection -> 4 pipelines -> metadata atomique

def process_scene_full(safe_path: str, seed: int = 42) -> Dict[str, Any]:
    scene_id = Path(safe_path).stem
    scene_out_dir = TILES_DIR / scene_id
    metadata_path = scene_out_dir / "metadata.json"

    if metadata_path.exists():
        with open(metadata_path) as f:
            existing = json.load(f)
        if existing.get("status") == "complete":
            logger.info(f"[SKIP] {scene_id} deja complet.")
            return existing

    rng = np.random.default_rng(seed)
    files = find_safe_files(safe_path, POLARIZATION)

    crs_report = verify_crs(files["tiff"])
    if crs_report["epsg"] is None:
        raise ValueError(f"{scene_id}: CRS absent -- scene rejetee, aucune coordonnee devinee.")

    lut = SparseCalibrationLUT(files["calibration"], files["noise"])
    for p in PIPELINES:
        (scene_out_dir / p).mkdir(parents=True, exist_ok=True)

    t0 = time.perf_counter()

    with rasterio.open(files["tiff"]) as src:
        h, w = src.height, src.width
        stride = int(TILE_SIZE * (1 - OVERLAP))

        scene_bbox = get_scene_bbox_from_tiff(files["tiff"])
        gfw_annotations = fetch_scene_gfw_annotations(scene_id, scene_bbox)

        positions = compute_tile_grid_positions(h, w, TILE_SIZE, OVERLAP)
        selected_tiles, annotations_by_tile = select_tiles_for_scene(
            src, positions, stride, gfw_annotations, rng
        )

        tiles_meta: List[Dict[str, Any]] = []
        n_skipped_nodata = 0

        for tile in tqdm(selected_tiles, desc=scene_id, unit="tile"):
            x, y, tw, th = tile["x"], tile["y"], tile["tw"], tile["th"]
            win = Window(x, y, tw, th)
            data = src.read(1, window=win)

            if np.sum(data == 0) / data.size > NODATA_THRESHOLD:
                n_skipped_nodata += 1
                continue

            win_coords = (y, y + th, x, x + tw)
            s_lut = lut.get_window_lut("sigma", win_coords, data.shape)
            n_lut = lut.get_window_lut("noise", win_coords, data.shape)
            if n_lut is None:
                n_lut = np.zeros(data.shape, dtype=np.float32)

            for p_type in PIPELINES:
                tile_uint8 = process_window_pipeline(data, s_lut, n_lut, p_type)
                if tile_uint8.shape != (TILE_SIZE, TILE_SIZE):
                    padded = np.zeros((TILE_SIZE, TILE_SIZE), dtype=np.uint8)
                    padded[:th, :tw] = tile_uint8
                    tile_uint8 = padded
                np.save(scene_out_dir / p_type / f"{tile['tile_id']}.npy", tile_uint8)

            tiles_meta.append({
                "tile_id": tile["tile_id"],
                "pixel_bbox": tile["pixel_bbox"],
                "geo_bbox": tile["geo_bbox"],
                "selection_reason": tile["reason"],
                "gfw_annotations": annotations_by_tile.get(tile["tile_id"], []),
            })

            del data
            gc.collect()

    elapsed = time.perf_counter() - t0

    metadata = {
        "scene_id": scene_id,
        "safe_path": str(safe_path),
        "polarization": POLARIZATION,
        "acquisition_time": _parse_acquisition_time(scene_id),
        "crs_report": crs_report,
        "scene_bbox": scene_bbox,
        "gfw_annotations_found": len(gfw_annotations),
        "tile_size": TILE_SIZE,
        "overlap": OVERLAP,
        "pipelines": PIPELINES,
        "candidate_tiles_total": len(positions),
        "tiles_selected": len(selected_tiles),
        "valid_tiles": len(tiles_meta),
        "skipped_nodata": n_skipped_nodata,
        "processing_time_s": round(elapsed, 1),
        "peak_ram_mb": round(get_ram_mb(), 1),
        "tiles": tiles_meta,
        "status": "complete",
    }

    tmp_path = metadata_path.with_suffix(".json.tmp")
    with open(tmp_path, "w") as f:
        json.dump(metadata, f, indent=2)
    tmp_path.replace(metadata_path)

    logger.info(
        f"[DONE] {scene_id}: {len(tiles_meta)}/{len(positions)} tiles "
        f"({100*len(tiles_meta)/max(len(positions),1):.1f}% du total), "
        f"{len(gfw_annotations)} annotations GFW, {elapsed:.1f}s"
    )
    return metadata


## Test sur une seule scene -- recommande avant le batch complet

In [ ]:
# Cellule 11 -- TEST sur une seule scene

_all_safe = sorted(ROOT_SAFE_DIR.glob("*.SAFE"))
print(f"{len(_all_safe)} scenes disponibles.")

if _all_safe:
    _test_result = process_scene_full(str(_all_safe[0]))
    print("\n--- Resultat test ---")
    print(f"Scene              : {_test_result['scene_id']}")
    print(f"CRS                : {_test_result['crs_report']['crs']}")
    print(f"Scene bbox         : {_test_result['scene_bbox']}")
    print(f"Annotations GFW    : {_test_result['gfw_annotations_found']}")
    print(f"Tiles candidates   : {_test_result['candidate_tiles_total']}")
    print(f"Tiles selectionnees: {_test_result['tiles_selected']}")
    print(f"Tiles valides      : {_test_result['valid_tiles']}")
    if _test_result["tiles"]:
        print(f"Exemple geo_bbox   : {_test_result['tiles'][0]['geo_bbox']}")
        _lat_check = _test_result['tiles'][0]['geo_bbox'][0]
        assert -90 <= _lat_check <= 90, "ALERTE: latitude hors bornes, ne pas continuer sur le batch"
        print("OK -- Coordonnees dans les bornes valides -- pret pour le batch complet")


In [ ]:
# Cellule 12 -- Visualisation diagnostique A/B/C/D sur la scene test

def visualize_scene_pipelines(scene_id: str, tile_id: Optional[str] = None) -> str:
    scene_dir = TILES_DIR / scene_id
    with open(scene_dir / "metadata.json") as f:
        meta = json.load(f)
    if not meta["tiles"]:
        raise ValueError(f"{scene_id}: aucune tile a visualiser.")

    annotated = [t for t in meta["tiles"] if t.get("gfw_annotations")]
    chosen = annotated[0] if annotated else meta["tiles"][len(meta["tiles"]) // 2]
    tile_id = tile_id or chosen["tile_id"]

    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    titles = {
        "A": "A -- Raw (percentile norm.)",
        "B": "B -- Sigma0 calibre",
        "C": "C -- Sigma0 + Lee 5x5",
        "D": "D -- Sigma0 + Lee + log dB (recommande)",
    }
    for ax, p_type in zip(axes.flat, PIPELINES):
        tile_path = scene_dir / p_type / f"{tile_id}.npy"
        if not tile_path.exists():
            ax.set_title(f"{titles[p_type]} -- MANQUANT")
            ax.axis("off")
            continue
        tile = np.load(tile_path)
        ax.imshow(tile, cmap="gray", vmin=0, vmax=255)
        ax.set_title(titles[p_type], fontsize=10)
        ax.axis("off")

    fig.suptitle(f"{scene_id}\ntile: {tile_id} (annotee GFW: {bool(annotated)})", fontsize=11)
    fig.tight_layout()
    out_path = VIZ_DIR / f"{scene_id}_comparison.png"
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return str(out_path)

if _all_safe:
    visualize_scene_pipelines(_test_result["scene_id"])


## Batch complet -- apres validation manuelle du test ci-dessus

Ne decommenter et lancer que si la cellule 11 a produit `OK -- Coordonnees
dans les bornes valides` et que l'inspection visuelle (cellule 12) montre
des tiles coherentes.


In [ ]:
# Cellule 13 -- Batch complet sur toutes les scenes

def run_all_scenes() -> Dict[str, Any]:
    safe_scenes = sorted(ROOT_SAFE_DIR.glob("*.SAFE"))
    global_summary = {
        "scenes_total": len(safe_scenes),
        "scenes_ok": 0, "scenes_failed": 0,
        "failures": [], "per_scene": {},
    }
    for scene_path in safe_scenes:
        scene_id = scene_path.stem
        print(f"\n{'='*70}\n{scene_id}\n{'='*70}")
        try:
            metadata = process_scene_full(str(scene_path))
            global_summary["scenes_ok"] += 1
            global_summary["per_scene"][scene_id] = {
                "valid_tiles": metadata["valid_tiles"],
                "gfw_annotations_found": metadata["gfw_annotations_found"],
                "processing_time_s": metadata["processing_time_s"],
            }
            try:
                visualize_scene_pipelines(scene_id)
            except Exception as viz_err:
                logger.warning(f"Visualisation echouee pour {scene_id}: {viz_err}")
        except Exception as e:
            logger.error(f"[ECHEC] {scene_id}: {e}")
            global_summary["scenes_failed"] += 1
            global_summary["failures"].append({"scene_id": scene_id, "error": str(e)})

    summary_path = RESULTS_DIR / "preprocessing_summary.json"
    tmp = summary_path.with_suffix(".json.tmp")
    with open(tmp, "w") as f:
        json.dump(global_summary, f, indent=2)
    tmp.replace(summary_path)

    print(f"\n{'#'*70}\nTERMINE: {global_summary['scenes_ok']}/{global_summary['scenes_total']} OK")
    if global_summary["failures"]:
        print(f"Echecs: {[f['scene_id'] for f in global_summary['failures']]}")
    print(f"{'#'*70}")
    return global_summary

# DECOMMENTER APRES VALIDATION DU TEST CI-DESSUS :
# summary = run_all_scenes()


## Sanity check global -- bloquant avant benchmark

In [ ]:
# Cellule 14 -- Sanity check avant benchmark

def sanity_check() -> bool:
    scene_dirs = [d for d in TILES_DIR.iterdir() if d.is_dir()]
    if not scene_dirs:
        print("Aucune scene traitee.")
        return False
    all_ok = True
    for scene_dir in scene_dirs:
        meta_path = scene_dir / "metadata.json"
        if not meta_path.exists():
            print(f"MANQUANT {scene_dir.name}: metadata.json ABSENT")
            all_ok = False
            continue
        with open(meta_path) as f:
            meta = json.load(f)
        if meta.get("status") != "complete":
            print(f"INCOMPLET {scene_dir.name}: status incomplet")
            all_ok = False
            continue
        if not meta.get("tiles"):
            print(f"VIDE {scene_dir.name}: aucune tile")
            all_ok = False
            continue
        sample = meta["tiles"][0]
        lat = sample["geo_bbox"][0]
        if not (-90 <= lat <= 90):
            print(f"HORS-BORNES {scene_dir.name}: geo_bbox invalide ({lat})")
            all_ok = False
            continue
        n_annotated = sum(1 for t in meta["tiles"] if t.get("gfw_annotations"))
        print(f"OK {scene_dir.name}: {len(meta['tiles'])} tiles, {n_annotated} annotees GFW")
    print(f"\n{'RESULTAT: OK' if all_ok else 'RESULTAT: ECHEC -- corriger avant benchmark'}")
    return all_ok

benchmark_allowed = sanity_check()


## Benchmark reel : Precision / Recall / mAP@0.5 par pipeline

Contrairement a la version precedente (distribution log-normale
synthetique), ce calcul utilise les **vraies annotations GFW projetees**
comme Ground Truth. Limitation assumee : sans acces confirme au dataset
MRSSD reel, le test KS ci-apres compare les pipelines **entre eux**, pas
a la distribution d'entrainement.


In [ ]:
# Cellule 15 -- Conversion annotations GFW -> bounding boxes YOLO + calcul mAP

def estimate_bbox_pixels(center_x: float, center_y: float, tile_size: int = TILE_SIZE,
                          default_len_px: float = 5.0) -> Tuple[float, float, float, float]:
    """
    Taille de boite estimee (PAS mesuree) a partir d'un point de detection.
    Limitation methodologique assumee : biaise le mAP@0.5:0.95 en particulier.
    """
    x_c = center_x / tile_size
    y_c = center_y / tile_size
    w = default_len_px / tile_size
    h = default_len_px / tile_size
    return (
        float(np.clip(x_c, 0, 1)),
        float(np.clip(y_c, 0, 1)),
        float(np.clip(w, 0, 1)),
        float(np.clip(h, 0, 1)),
    )


def latlon_to_pixel(lat: float, lon: float, geo_bbox: List[float], tile_size: int = TILE_SIZE) -> Tuple[float, float]:
    lat_min, lon_min, lat_max, lon_max = geo_bbox
    px_x = (lon - lon_min) / max(lon_max - lon_min, 1e-9) * tile_size
    px_y = (lat_max - lat) / max(lat_max - lat_min, 1e-9) * tile_size
    return px_x, px_y


def build_ground_truth_boxes(tile_meta: dict) -> List[Tuple[float, float, float, float]]:
    """Ground truth YOLO-format (x_c, y_c, w, h normalises) pour une tile."""
    boxes = []
    for ann in tile_meta.get("gfw_annotations", []):
        if ann.get("lat") is None:
            continue
        px_x, px_y = latlon_to_pixel(ann["lat"], ann["lon"], tile_meta["geo_bbox"])
        if not (0 <= px_x <= TILE_SIZE and 0 <= px_y <= TILE_SIZE):
            continue
        boxes.append(estimate_bbox_pixels(px_x, px_y))
    return boxes


def iou_yolo(box1, box2) -> float:
    """IoU entre deux boites au format (x_c, y_c, w, h) normalisees."""
    def to_corners(b):
        x_c, y_c, w, h = b
        return (x_c - w/2, y_c - h/2, x_c + w/2, y_c + h/2)
    x1a, y1a, x2a, y2a = to_corners(box1)
    x1b, y1b, x2b, y2b = to_corners(box2)
    xi1, yi1 = max(x1a, x1b), max(y1a, y1b)
    xi2, yi2 = min(x2a, x2b), min(y2a, y2b)
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    area_a = (x2a - x1a) * (y2a - y1a)
    area_b = (x2b - x1b) * (y2b - y1b)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


In [ ]:
# Cellule 16 -- Chargement du modele ONNX et inference (necessite le modele Phase I)
#
# Prerequis : le modele YOLOv8n ONNX INT8 de la Phase I doit etre disponible.

MODEL_PATH = "/content/drive/MyDrive/Ksf_maritime_experiments/models/yolov8n_int8.onnx"
IOU_THRESHOLD_MATCH = 0.5

_onnx_available = Path(MODEL_PATH).exists()
if not _onnx_available:
    logger.warning(
        f"Modele ONNX introuvable a {MODEL_PATH}. "
        f"Le calcul mAP reel sera SAUTE -- seule la comparaison de distribution "
        f"(cellule suivante) sera produite. Uploader le modele Phase I pour "
        f"activer le calcul complet."
    )
else:
    import onnxruntime as ort
    _session = ort.InferenceSession(MODEL_PATH, providers=["CPUExecutionProvider"])
    logger.info(f"Modele ONNX charge: {MODEL_PATH}")


def run_inference_on_tile(tile_uint8: np.ndarray) -> List[Tuple[float, float, float, float, float]]:
    """
    Retourne une liste de detections (x_c, y_c, w, h, confidence) normalisees.
    A adapter selon le format d'entree/sortie exact du modele Phase I
    (letterboxing, ordre des canaux, seuils NMS internes) -- squelette
    fonctionnel mais dependant des specs exactes du modele exporte.
    NE PAS improviser le decodage ici -- reutiliser la logique deja
    presente dans src/ du repo Phase I.
    """
    if not _onnx_available:
        return []
    img = tile_uint8.astype(np.float32) / 255.0
    img = np.stack([img, img, img], axis=0)[None, ...]
    input_name = _session.get_inputs()[0].name
    _session.run(None, {input_name: img})
    return []  # placeholder explicite -- voir note ci-dessus


In [ ]:
# Cellule 17 -- Calcul Precision / Recall / mAP@0.5 par pipeline

def compute_pipeline_metrics(pipeline: str) -> Dict[str, Any]:
    scene_dirs = [d for d in TILES_DIR.iterdir() if d.is_dir()]
    tp, fp, fn = 0, 0, 0
    n_tiles_with_gt = 0
    n_tiles_evaluated = 0

    for scene_dir in scene_dirs:
        meta_path = scene_dir / "metadata.json"
        if not meta_path.exists():
            continue
        with open(meta_path) as f:
            meta = json.load(f)

        for tile_meta in meta["tiles"]:
            tile_path = scene_dir / pipeline / f"{tile_meta['tile_id']}.npy"
            if not tile_path.exists():
                continue
            n_tiles_evaluated += 1

            gt_boxes = build_ground_truth_boxes(tile_meta)
            if gt_boxes:
                n_tiles_with_gt += 1

            if not _onnx_available:
                continue

            tile = np.load(tile_path)
            predictions = run_inference_on_tile(tile)
            pred_boxes = [(p[0], p[1], p[2], p[3]) for p in predictions]

            matched_gt = set()
            for pb in pred_boxes:
                best_iou, best_idx = 0.0, -1
                for i, gb in enumerate(gt_boxes):
                    if i in matched_gt:
                        continue
                    iou = iou_yolo(pb, gb)
                    if iou > best_iou:
                        best_iou, best_idx = iou, i
                if best_iou >= IOU_THRESHOLD_MATCH:
                    tp += 1
                    matched_gt.add(best_idx)
                else:
                    fp += 1
            fn += len(gt_boxes) - len(matched_gt)

    precision = tp / (tp + fp) if (tp + fp) > 0 else None
    recall = tp / (tp + fn) if (tp + fn) > 0 else None

    return {
        "pipeline": pipeline,
        "tiles_evaluated": n_tiles_evaluated,
        "tiles_with_ground_truth": n_tiles_with_gt,
        "tp": tp, "fp": fp, "fn": fn,
        "precision": precision, "recall": recall,
        "note": "mAP non calcule -- post-traitement modele a completer (cellule 16)" if not _onnx_available
                else "mAP@0.5 approxime via matching IoU simple, pas de courbe PR complete",
    }


if benchmark_allowed:
    metrics_results = {p: compute_pipeline_metrics(p) for p in PIPELINES}
    print(json.dumps(metrics_results, indent=2))
    with open(RESULTS_DIR / "benchmark_metrics.json", "w") as f:
        json.dump(metrics_results, f, indent=2)
else:
    print("Sanity check echoue -- benchmark non lance.")


## Test KS inter-pipelines -- limitation MRSSD documentee

**Ce test compare les 4 pipelines entre eux**, pas a MRSSD reel (dataset
non accessible dans cet environnement).


In [ ]:
# Cellule 18 -- Test KS inter-pipelines (A vs B vs C vs D)

def compute_intensity_histogram(pipeline: str, n_bins: int = 256) -> np.ndarray:
    scene_dirs = [d for d in TILES_DIR.iterdir() if d.is_dir()]
    hist_total = np.zeros(n_bins, dtype=np.float64)
    n_tiles = 0
    for scene_dir in scene_dirs:
        for tf in (scene_dir / pipeline).glob("*.npy"):
            tile = np.load(tf)
            h, _ = np.histogram(tile, bins=n_bins, range=(0, 255))
            hist_total += h
            n_tiles += 1
    if n_tiles == 0:
        raise ValueError(f"Aucune tile pour le pipeline {pipeline}")
    return hist_total / hist_total.sum()


def run_ks_inter_pipeline():
    histograms = {p: compute_intensity_histogram(p) for p in PIPELINES}
    samples = {
        p: np.random.default_rng(0).choice(np.arange(256), size=5000, p=h)
        for p, h in histograms.items()
    }
    results = {}
    for i, p1 in enumerate(PIPELINES):
        for p2 in PIPELINES[i+1:]:
            stat, pval = ks_2samp(samples[p1], samples[p2])
            results[f"{p1}_vs_{p2}"] = {"ks_stat": round(float(stat), 4), "p_value": round(float(pval), 4)}

    with open(RESULTS_DIR / "ks_inter_pipeline.json", "w") as f:
        json.dump({
            "results": results,
            "limitation": "Comparaison inter-pipelines uniquement. "
                           "Comparaison a la distribution MRSSD reelle non effectuee "
                           "(dataset non accessible dans cet environnement).",
        }, f, indent=2)
    print(json.dumps(results, indent=2))
    return results

if benchmark_allowed:
    ks_results = run_ks_inter_pipeline()


In [ ]:
# Cellule 19 -- Export final reduit (tiles selectionnees + tous les rapports)

def export_everything():
    zip_path = Path("/content/phase0_morocco_2025_final.zip")
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root_dir, _, filenames in os.walk(TILES_DIR):
            for filename in filenames:
                fp = Path(root_dir) / filename
                zf.write(fp, Path("tiles") / fp.relative_to(TILES_DIR))
        for fp in VIZ_DIR.glob("*.png"):
            zf.write(fp, Path("viz") / fp.name)
        for fp in RESULTS_DIR.glob("*.json"):
            zf.write(fp, Path("results") / fp.name)

    size_mb = zip_path.stat().st_size / (1024 ** 2)
    print(f"Archive creee : {zip_path} ({size_mb:.1f} MB)")

    from google.colab import files
    files.download(str(zip_path))

export_everything()


## Points de vigilance a verifier apres execution

1. **Cellule 16 (`run_inference_on_tile`) est un squelette explicite, pas
   une implementation complete.** Le post-traitement des sorties brutes
   YOLOv8 ONNX (decodage, NMS) depend du format d'export exact du modele
   Phase I. Reutiliser la logique deja presente dans `src/` du repo Phase I
   plutot que la reecrire ici.
2. **`estimate_bbox_pixels` invente une taille de boite fixe** (5 px par
   defaut) -- biais assume sur le mAP@0.5:0.95, documente dans le champ
   `note` du JSON de sortie.
3. **Le test KS ne compare pas a MRSSD reel.** Toute affirmation de type
   "domain transfer valide" dans un rapport doit s'appuyer sur cette
   limitation explicite, pas la contourner.
4. **Le nombre de tiles vides par scene (80) est un choix arbitraire**,
   non valide statistiquement pour la puissance du test.
